# Пример запуска: задача про монеты Туг-туг

Ноутбук показывает полный локальный сценарий: создать файл задачи, проверить секреты, запустить выбранные модели через `runner.py` и прочитать ответы из JSON-лога.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'runner.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(ROOT)

## 1. Создать файл задачи

Задача будет записана в `data/problems/tug_tug_500.json`.

In [ ]:
problem = {
    'id': 'tug_tug_500',
    'title': 'Монеты страны Туг-туг',
    'source': 'User-provided olympiad problem',
    'text': (
        'В стране Туг-туг в ходу монеты, номиналы которых равны 1, 2 и 3 тугрика. '
        'Однажды Нетуг, житель страны Туг-туг, нашел у себя в кармане 500 монет '
        'суммарным номиналом 1000 тугриков. Докажите, что он может набрать '
        '500 тугриков без сдачи.'
    ),
    'expected_answer': None,
}

problem_path = Path('data/problems/tug_tug_500.json')
problem_path.parent.mkdir(parents=True, exist_ok=True)
problem_path.write_text(json.dumps(problem, ensure_ascii=False, indent=2), encoding='utf-8')
print(problem_path)
print(problem['text'])

## 2. Выбрать модели

По умолчанию запускаются `gpt,gigachat,yandexgpt`. Если какой-то провайдер не нужен, уберите его из переменной `MODELS`.

In [ ]:
MODELS = 'gpt,gigachat,yandexgpt'
RUN_ID = 'tug_tug_all_models'
print(MODELS, RUN_ID)

## 3. Проверить секреты

Команда не печатает значения ключей.

In [ ]:
!python scripts/check_secrets.py --models $MODELS

## 4. Запустить модели

Основной интерфейс проекта - `runner.py`.

In [ ]:
!python runner.py --problem data/problems/tug_tug_500.json --models $MODELS --run-id $RUN_ID

## 5. Прочитать лог и ответы

Ниже выводятся только полезные поля. Полный raw response остается в JSON-логе.

In [ ]:
log_path = Path('logs') / f'{RUN_ID}.json'
log = json.loads(log_path.read_text(encoding='utf-8'))

for result in log['results']:
    print('=' * 80)
    print('model:', result['model'])
    print('status:', 'error' if result['error'] else 'ok')
    print('tokens:', result['prompt_tokens'] + result['completion_tokens'])
    print('cost_usd:', result['cost_usd'])
    print('latency_ms:', result['latency_ms'])
    if result['error']:
        print('error:', result['error'])
    else:
        print(result['answer'])

## 6. Открыть ручной скоринг

После запуска можно открыть веб-интерфейс и оценить ответы.

```bash
python scoring/app.py
```

Затем открыть `http://127.0.0.1:8000/run/tug_tug_all_models`.